In [45]:
import os
import cv2
import numpy as np
import pandas as pd
import librosa
from moviepy.editor import VideoFileClip
from keras.applications import ResNet50
from keras.models import Model
from keras.layers import Conv2D, LSTM, Dense, Flatten, Concatenate, Input, Dropout
from keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import dlib

In [46]:
# Load dlib's face detector and face landmarks predictor for facial expressions and gaze
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor('/Users/jeremiah/video-p-reaction-joketype-modular/Cursor_october/dataa/shape_predictor_68_face_landmarks.dat')

In [47]:
# Function to extract optical flow (movement) between consecutive frames
def extract_optical_flow(frames):
    optical_flows = []
    prev_frame = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)
    for i in range(1, len(frames)):
        next_frame = cv2.cvtColor(frames[i], cv2.COLOR_BGR2GRAY)
        flow = cv2.calcOpticalFlowFarneback(prev_frame, next_frame, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        optical_flows.append(flow)
        prev_frame = next_frame
    return np.array(optical_flows)


# Function to extract visual features using ResNet50
def extract_visual_features(frames, resnet_model):
    features = [resnet_model.predict(np.expand_dims(frame, axis=0)) for frame in frames]
    return np.array(features)



# Function to extract facial expressions and gaze features using dlib
def extract_facial_features(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = detector(gray)
    if len(faces) > 0:
        landmarks = predictor(gray, faces[0])
        facial_features = [(landmarks.part(n).x, landmarks.part(n).y) for n in range(68)]
        return np.array(facial_features)
    return np.zeros((68, 2))

# Function to extract audio from a video
def extract_audio(video_path):
    clip = VideoFileClip(video_path)
    audio = clip.audio
    audio_path = video_path.replace(".mp4", ".wav")
    audio.write_audiofile(audio_path)
    return audio_path

# Function to extract MFCC features from audio
def extract_mfcc(audio_path, n_mfcc=13):
    y, sr = librosa.load(audio_path)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    return mfccs



# Function to extract gesture quality metrics (e.g., hands movement)
def extract_gesture_quality(frames):
    # Use pre-trained gesture recognition models, or hand-crafted gesture quality features
    # For now, we'll just simulate the extraction of gesture features as an example
    # In practice, you can implement a model that analyzes hand movement and pose detection
    gesture_metrics = np.random.rand(len(frames), 5)  # 5 random gesture features per frame
    return gesture_metrics

# Function to sample frames from a video and extract motion, gaze, facial expressions
def sample_frames_and_extract_features(video_path, n_frames=10):

    cap = cv2.VideoCapture(video_path)
    frames = []
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    n_frames_to_sample = min(n_frames, frame_count)
    
    optical_flows, facial_features, gesture_features = [], [], []
    
    for i in range(n_frames_to_sample):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_count * i / n_frames_to_sample))
        ret, frame = cap.read()
        if ret:
            frame = cv2.resize(frame, (224, 224))  # Resize to a consistent shape
            frames.append(frame)
            optical_flows.append(extract_optical_flow([frame]))
            facial_features.append(extract_facial_features(frame))
            gesture_features.append(extract_gesture_quality([frame]))
    
    cap.release()
    return np.array(frames), np.array(optical_flows), np.array(facial_features), np.array(gesture_features)

#########################

def sample_frames_and_extract_features2(video_path, n_frames=10):
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"Error: Cannot open video {video_path}")
        return [], [], [], []  # Return empty lists if video cannot be opened
    
    frames = []
    optical_flows, facial_features, gesture_features = [], [], []
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    n_frames_to_sample = min(n_frames, frame_count)
    
    for i in range(n_frames_to_sample):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_count * i / n_frames_to_sample))
        ret, frame = cap.read()
        if ret:
            frame = cv2.resize(frame, (224, 224))  # Resize to a consistent shape
            frames.append(frame)

            # Extract features only if the frame is valid
            optical_flows.append(extract_optical_flow([frame]))
            facial_feature = extract_facial_features(frame)
            gesture_feature = extract_gesture_quality([frame])

            # Check if facial features were found
            if facial_feature.size == 0:
                facial_features.append(np.zeros((68, 2)))  # Append a zero array if no face detected
            else:
                facial_features.append(facial_feature)

            gesture_features.append(gesture_feature)
        else:
            print(f"Warning: Frame not found at position {i}")

    cap.release()
    
    if not frames:  # Check if any frames were successfully captured
        print("Error: No frames captured.")
        return [], [], [], []
    
    return np.array(frames), np.array(optical_flows), np.array(facial_features), np.array(gesture_features)

def preprocess_video_data2(video_dir, metadata, resnet_model):
    
    video_features_list, optical_flow_list = [], []
    facial_features_list, gesture_features_list = []
    audio_features_list, labels = []
    metadata_features_list = []
    
    for index, row in metadata.iterrows():
        video_id = row['video_id']
        video_path = os.path.join(video_dir, f'{video_id}.mp4')
        
        # Sample frames and extract features
        video_frames, optical_flows, facial_features, gesture_features = sample_frames_and_extract_features2(video_path, n_frames=10)
        
        # Check if any features were extracted
        if len(video_frames) == 0:
            print(f"Warning: No features extracted for video {video_id}. Skipping this video.")
            continue  # Skip this video if no frames were captured
        
        # Extract visual features using ResNet50
        video_features = extract_visual_features(video_frames, resnet_model)
        video_features_list.append(video_features)
        
        # Append extracted features
        optical_flow_list.append(optical_flows)
        facial_features_list.append(facial_features)
        gesture_features_list.append(gesture_features)
        
        # Extract audio features
        audio_path = extract_audio(video_path)
        audio_features = extract_mfcc(audio_path)
        audio_features_list.append(audio_features)
        
        # Extract relevant metadata features (e.g., joke type, laughter, etc.)
        metadata_features = [row['JokeType_encoded'], row['Laugh_encoded'], row['Gender_encoded'], row['Age']]
        metadata_features_list.append(metadata_features)
        
        # Append the label (how funny)
        labels.append(row['how_funny_encoded'])
    
    # Ensure all lists are converted to arrays only if they contain data
    return (np.array(video_features_list), np.array(optical_flow_list),
            np.array(facial_features_list), np.array(gesture_features_list),
            np.array(audio_features_list), np.array(metadata_features_list), np.array(labels))

#########

# Function to load and preprocess video and audio data, and also extract metadata
def preprocess_video_data(video_dir, metadata, resnet_model):

    video_features_list, optical_flow_list = [], []
    facial_features_list, gesture_features_list = []
    audio_features_list, labels = []
    metadata_features_list = []
    
    for index, row in metadata.iterrows():
        video_id = row['VideoID']
        video_path = os.path.join(video_dir, f'{video_id}.mp4')
        
        # Sample frames and extract features
        video_frames, optical_flows, facial_features, gesture_features = sample_frames_and_extract_features(video_path, n_frames=10)
        
        # Extract visual features using ResNet50
        video_features = extract_visual_features(video_frames, resnet_model)
        video_features_list.append(video_features)
        
        # Append extracted features
        optical_flow_list.append(optical_flows)
        facial_features_list.append(facial_features)
        gesture_features_list.append(gesture_features)
        
        # Extract audio features
        audio_path = extract_audio(video_path)
        audio_features = extract_mfcc(audio_path)
        audio_features_list.append(audio_features)
        
        # Extract relevant metadata features (e.g., joke type, laughter, etc.)
        metadata_features = [row['JokeType_encoded'], row['Laugh_encoded'], row['Gender_encoded'], row['Age']]
        metadata_features_list.append(metadata_features)
        
        # Append the label (how funny)
        labels.append(row['HowFunny_encoded'])
    
    return (np.array(video_features_list), np.array(optical_flow_list),
            np.array(facial_features_list), np.array(gesture_features_list),
            np.array(audio_features_list), np.array(metadata_features_list), np.array(labels))

# Function to build a multi-modal model with intermediate fusion
def build_multimodal_model():
    # Video input (pre-extracted visual features)
    video_input = Input(shape=(10, 7, 7, 2048))  # ResNet50 output shape
    x1 = Flatten()(video_input)
    
    # Optical flow (movement) input
    optical_flow_input = Input(shape=(10, 224, 224, 2))  # Optical flow between consecutive frames
    x2 = Conv2D(32, (3, 3), activation='relu')(optical_flow_input)
    x2 = Flatten()(x2)
    
    # Facial features input
    facial_input = Input(shape=(10, 68, 2))  # Facial landmarks (68 points)
    x3 = Flatten()(facial_input)
    
    # Gesture features input
    gesture_input = Input(shape=(10, 5))  # Gesture quality features
    x4 = LSTM(64)(gesture_input)
    
    # Audio input (MFCC features)
    audio_input = Input(shape=(13, None))  # Assuming variable-length audio features
    x5 = LSTM(64)(audio_input)
    
    # Metadata input (e.g., joke type, interaction quality, laughter)
    metadata_input = Input(shape=(3,))  # Metadata features
    x6 = Dense(32, activation='relu')(metadata_input)
    
    # Concatenate all extracted features (intermediate fusion)
    combined = Concatenate()([x1, x2, x3, x4, x5, x6])
    
    # Output layer (4 classes for how funny)
    output = Dense(4, activation='softmax')(combined)
    
    # Model definition
    model = Model(inputs=[video_input, optical_flow_input, facial_input, gesture_input, audio_input, metadata_input], outputs=output)
    model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])
    
    return model

# Function to train and evaluate the model
def train_and_evaluate_model(video_features, optical_flows, facial_features, gesture_features, audio_features, metadata_features, labels):
    # Split the data into training and testing sets
    X_train_video, X_test_video, X_train_optical, X_test_optical, X_train_facial, X_test_facial = train_test_split(
        video_features, optical_flows, facial_features, test_size=0.2, random_state=42)
    
    X_train_gesture, X_test_gesture, X_train_audio, X_test_audio, X_train_metadata, X_test_metadata, y_train, y_test = train_test_split(
        gesture_features, audio_features, metadata_features, labels, test_size=0.2, random_state=42)
    
    # Build the multi-modal model
    model = build_multimodal_model()
    
    # Train the model
    model.fit([X_train_video, X_train_optical, X_train_facial, X_train_gesture, X_train_audio, X_train_metadata],
              y_train, epochs=10, batch_size=16, validation_split=0.2)
    
    # Predict and evaluate on the test set
    y_pred = model.predict([X_test_video, X_test_optical, X_test_facial, X_test_gesture, X_test_audio, X_test_metadata])
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_test_classes = np.argmax(y_test, axis=1)
    
    # Print evaluation metrics
    accuracy = accuracy_score(y_test_classes, y_pred_classes)
    f1 = f1_score(y_test_classes, y_pred_classes, average='weighted')
    conf_matrix = confusion_matrix(y_test_classes, y_pred_classes)
    
    print(f'Accuracy: {accuracy:.2f}')
    print(f'F1 Score: {f1:.2f}')
    print(f'Confusion Matrix:\n{conf_matrix}')
    
    return model

In [48]:
# Load the ResNet50 model for feature extraction
resnet_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

In [49]:
# Example usage
excel_path = '/Users/jeremiah/video-p-reaction-joketype-modular/Cursor_october/dataa/excel.xlsx'
video_path = '/Users/jeremiah/video-p-reaction-joketype-modular/Cursor_october/dataa/fifteen_twenty_five'
metadata = pd.read_excel(excel_path)

In [50]:
# Encode categorical labels
le = LabelEncoder()
metadata['Laugh_encoded'] = le.fit_transform(metadata['Laugh'])
metadata['JokeType_encoded'] = le.fit_transform(metadata['JokeType'])
metadata['HowFunny_encoded'] = le.fit_transform(metadata['HowFunny'])
metadata['Gender_encoded'] = le.fit_transform(metadata['Gender'])

metadata = metadata[['VideoID', 'Age', 'Gender_encoded', 'Laugh_encoded', 'JokeType_encoded', 'HowFunny_encoded']]

In [51]:
metadata.head()

,VideoID,Age,Gender_encoded,Laugh_encoded,JokeType_encoded,HowFunny_encoded
0,NFCK3A.joke1.rep1.take1.ThatsNotAHat.mp4,109,0,0,4,3
1,NFCK3A.joke1.rep1.take2.ThatsNotAHat.mp4,109,0,0,4,3
2,NFCK3A.joke1.rep2.take1.ThatsNotAHat.mp4,109,0,0,4,2
3,NFCK3A.joke1.rep3.take1.ThatsNotAHat.mp4,109,0,0,4,2
4,NFCK3A.joke2.rep1.take1.TearingPaper.mp4,109,0,0,2,2


In [52]:
# Preprocess video data and metadata
video_features, optical_flows, facial_features, gesture_features, audio_features, metadata_features, labels = preprocess_video_data2(video_path, 
                                                                                                                                    metadata, 
                                                                                                                                    resnet_model)

ValueError: not enough values to unpack (expected 2, got 0)

In [21]:
# Train the model
model = train_and_evaluate_model(video_features, optical_flows, facial_features, gesture_features, audio_features, metadata_features, labels)


NameError: name 'video_features' is not defined